In [6]:
import sys
import json

sys.path.append("../src")

from langchain_core.utils.function_calling import convert_to_openai_tool
from mortgage_agent.retrieval import retrieve
from mortgage_agent.tools import ltv, dti, funds_to_close, reserves_months, add_years, search_guide

In [2]:
print(type(ltv))
print(ltv.name)
print(ltv.description)
print(ltv.args)
print(ltv.invoke({"loan_amount": 320000, "property_value": 400000}))

<class 'langchain_core.tools.structured.StructuredTool'>
ltv
Loan-to-value ratio as a percentage: loan amount divided by property value, times 100.
{'loan_amount': {'title': 'Loan Amount', 'type': 'number'}, 'property_value': {'title': 'Property Value', 'type': 'number'}}
80.0


In [3]:
print(json.dumps(convert_to_openai_tool(ltv), indent=2))

{
  "type": "function",
  "function": {
    "name": "ltv",
    "description": "Loan-to-value ratio as a percentage: loan amount divided by property value, times 100.",
    "parameters": {
      "properties": {
        "loan_amount": {
          "type": "number"
        },
        "property_value": {
          "type": "number"
        }
      },
      "required": [
        "loan_amount",
        "property_value"
      ],
      "type": "object"
    }
  }
}


In [4]:
print(dti.args)
print(dti.invoke({"housing_payment": 2420, "monthly_debts": [450, 120], "monthly_income": 8500}))
print(funds_to_close.invoke({"property_value": 400000, "loan_amount": 320000, "closing_costs": 9000}))
print(reserves_months.invoke({"available_funds": 55000, "cash_to_close": 89000, "monthly_pitia": 2420}))
print(reserves_months.invoke({"available_funds": 95000, "cash_to_close": 89000, "monthly_pitia": 2420}))
print(add_years.invoke({"start_date": "2022-10-30", "years": 4}))
print(add_years.invoke({"start_date": "2024-02-29", "years": 1}))

{'housing_payment': {'title': 'Housing Payment', 'type': 'number'}, 'monthly_debts': {'items': {'type': 'number'}, 'title': 'Monthly Debts', 'type': 'array'}, 'monthly_income': {'title': 'Monthly Income', 'type': 'number'}}
35.18
89000.0
-14.05
2.48
2026-10-30
2025-02-28


In [5]:
from mortgage_agent.retrieval import retrieve

for q in ["large deposit documentation", "Chapter 7 bankruptcy waiting period", "gift from parents"]:
    print(q, "→", [d.metadata["rule_id"] for d in retrieve(q, top_n=3)])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

large deposit documentation → ['B3-4.2-02', 'B3-4.2-02', 'B3-4.2-02']
Chapter 7 bankruptcy waiting period → ['B3-5.3-07', 'B3-5.3-07', 'B3-5.3-07']
gift from parents → ['B3-4.3-04', 'B3-4.3-04', 'B3-4.3-05']


In [7]:
result = search_guide.invoke({"query": "large deposit documentation"})
print(len(result), "characters")
print(result[:700])

3797 characters
Fannie Mae Selling Guide, B3-4.2-02: Depository Accounts

loan.  When bank statements (typically covering the most recent two months) are used, the lender must
evaluate large deposits. See B3-4.1-04, Virtual Currency for additional information when a large deposit may be
from virtual currency that was exchanged into U.S. dollars. Requirements for evaluating large deposits vary
based on the transaction type, as shown in the table below.
Transaction Type
Evaluation Requirements
Refinance transactions
Documentation or explanation for large deposits is not required; however,
the lender remains responsible for ensuring that any borrowed funds,
including any related liability, are considered.
Tran
